# MUP by-Provider-and-Service — Delaware peek

Scope: per-NPI x HCPCS billing detail (what a provider bills). Core for Q2 (top services), Q5 (service mix).
Source: `data/sources/utilization_and_payment/mup/Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2024.zip` (~3.3 GB).
Reading: chunked pandas, filtered to `Rndrng_Prvdr_State_Abrvtn` = DE.

In [1]:
import zipfile
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ZIP = ROOT / "data/sources/utilization_and_payment/mup/Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2024.zip"
MEMBER = "Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2024.csv"
STATE_COL, STATE, ENC = "Rndrng_Prvdr_State_Abrvtn", "DE", "latin-1"
KEEP = [
    "Rndrng_NPI", "Rndrng_Prvdr_State_Abrvtn", "Rndrng_Prvdr_Type",
    "HCPCS_Cd", "HCPCS_Desc", "HCPCS_Drug_Ind", "Place_Of_Srvc",
    "Tot_Benes", "Tot_Srvcs", "Avg_Mdcr_Pymt_Amt",
]
with zipfile.ZipFile(ZIP) as z, z.open(MEMBER) as f:
    sample = pd.read_csv(f, nrows=5, dtype=str, encoding=ENC)
print(sample.shape[1], "columns")
sample[KEEP].head()

28 columns


,Rndrng_NPI,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_Type,HCPCS_Cd,HCPCS_Desc,HCPCS_Drug_Ind,Place_Of_Srvc,Tot_Benes,Tot_Srvcs,Avg_Mdcr_Pymt_Amt
0,1003000126,MD,Internal Medicine,99221,Initial hospital care with straightforward or ...,N,F,36,36,60.828888889
1,1003000126,MD,Internal Medicine,99222,Initial hospital care with straightforward or ...,N,F,150,150,95.6756
2,1003000126,MD,Internal Medicine,99223,Initial hospital care with moderate level of m...,N,F,63,63,134.30047619
3,1003000126,MD,Internal Medicine,99231,Subsequent hospital care with straightforward ...,N,F,11,16,33.350625
4,1003000126,MD,Internal Medicine,99232,Subsequent hospital care with moderate levelof...,N,F,37,70,62.728714286


In [2]:
# Scope to Delaware. Chunked scan (~3.3 GB, ~10M rows nationally), key columns only.
parts = []
with zipfile.ZipFile(ZIP) as z, z.open(MEMBER) as f:
    for chunk in pd.read_csv(f, usecols=KEEP, dtype=str, chunksize=500_000, encoding=ENC):
        parts.append(chunk[chunk[STATE_COL] == STATE])
de = pd.concat(parts, ignore_index=True)
de.shape

(40082, 10)

In [3]:
# Service mix: rows per provider, most common billed codes.
print("rows:", len(de), "| unique NPIs:", de["Rndrng_NPI"].nunique(), "| unique HCPCS:", de["HCPCS_Cd"].nunique())
print("rows per NPI:", round(len(de) / de["Rndrng_NPI"].nunique(), 1))
de["HCPCS_Desc"].value_counts().head(15)

rows: 40082 | unique NPIs: 4680 | unique HCPCS: 1837
rows per NPI: 8.6


HCPCS_Desc
Established patient office or other outpatient visit with moderate level of decision making, if using time, 30 minutes or more                                                                                                                                      2059
Established patient office or other outpatient visit with low level od decision making, if using time, 20 minutes or more                                                                                                                                           1921
New patient office or other outpatient visit with moderate level of medical decision making, if using time, 45 minutes or more                                                                                                                                      1029
Subsequent hospital care with moderate levelof medical decision making, if using time, at least 35 minutes                                                                                        

## Notes
- Grain: one row per NPI x HCPCS x place-of-service. Dollars are **averages** (`Avg_Mdcr_Pymt_Amt`), not totals.
- Service total spend = `Avg_Mdcr_Pymt_Amt` * `Tot_Srvcs`. Rank top-N services by that or by `Tot_Srvcs`.
- Place-of-service splits a provider+code into F (facility) and O (office); a code can appear twice.
- Suppression: rows with < 11 beneficiaries are dropped upstream by CMS.